# SkillSight: Job Market Analysis & Technical Skill Intelligence System

**Module 8: Comprehensive Jupyter Notebook Documentation**

---

### Abstract
SkillSight is an end-to-end data analytics and machine learning system developed to extract actionable labor market intelligence from scraped LinkedIn job postings (15,886 records). The system implements a rigorous data science workflow comprising:
1. **Data Preprocessing**: Data cleaning, text normalization, deduplication, and compensation alignment (`src/preprocessing.py`).
2. **Technical Skill Extraction**: Explainable boundary-safe regex taxonomy covering 49 in-demand technologies (`src/skill_extraction.py`).
3. **Exploratory Data Analysis (EDA)**: Profiling job titles, locations, companies, seniority levels, work types, and salary distributions (`src/analysis.py`).
4. **Data Visualization**: High-resolution publication-standard charts saved to `outputs/charts/` (`src/visualization.py`).
5. **Machine Learning**: Supervised salary regression via Random Forest with complete serialized pipeline (`src/ml_model.py`).
6. **Interactive Dashboard**: Modern Streamlit academic presentation interface (`dashboard/app.py`).

```text
Cleaned Dataset -> Feature Extraction -> EDA & Visualization -> Machine Learning -> Streamlit Dashboard
```


## 1. Import Required Libraries

We import core scientific computing, data manipulation, and visualization libraries (`pandas`, `numpy`, `matplotlib`, `joblib`).


In [3]:
import os
import sys
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use('default')
print("Python Version:", sys.version.split()[0])
print("Pandas Version:", pd.__version__)
print("NumPy Version:", np.__version__)
print("Joblib Version:", joblib.__version__)


Python Version: 3.13.15
Pandas Version: 3.0.5
NumPy Version: 2.5.2
Joblib Version: 1.5.3


## 2. Load Dataset

We load the preprocessed and cleaned job postings dataset from `data/cleaned_jobs.csv`.


In [5]:
DATA_PATH = "../data/cleaned_jobs.csv"
df_jobs = pd.read_csv(DATA_PATH)
print(f"Successfully loaded dataset: {len(df_jobs):,} job postings, {df_jobs.shape[1]} columns.")


Successfully loaded dataset: 15,886 job postings, 23 columns.


## 3. Dataset Overview

An inspection of the structural dimensions, column data types, and sample records.


In [7]:
print("Dataset Dimensions:")
print(f"- Total Rows:    {df_jobs.shape[0]:,}")
print(f"- Total Columns: {df_jobs.shape[1]}")

print("\nColumn Data Types & Non-Null Counts:")
overview_df = pd.DataFrame({
    "Data Type": df_jobs.dtypes,
    "Non-Null Count": df_jobs.notnull().sum(),
    "Null Count": df_jobs.isnull().sum()
})
print(overview_df.to_string())

print("\nFirst 5 Sample Records (Selected Columns):")
sample_cols = ["job_id", "title", "location", "formatted_experience_level", "formatted_work_type", "salary", "remote_status"]
print(df_jobs[sample_cols].head(5).to_string(index=False))


Dataset Dimensions:
- Total Rows:    15,886
- Total Columns: 23

Column Data Types & Non-Null Counts:
                           Data Type  Non-Null Count  Null Count
job_id                         int64           15886           0
company_id                   float64           15520         366
title                            str           15886           0
description                      str           15885           1
min_salary                   float64            5521       10365
med_salary                   float64             981       14905
max_salary                   float64            5521       10365
salary                       float64            6502        9384
has_salary                      bool           15886           0
pay_period                       str            6502        9384
formatted_work_type              str           15886           0
location                         str           15886           0
applies                      float64           15886 

## 4. Data Quality Analysis

Evaluating missing value rates across columns and summarizing engagement/compensation metrics.


In [9]:
missing_df = pd.DataFrame({
    "Missing Count": df_jobs.isnull().sum(),
    "Missing Percentage (%)": (df_jobs.isnull().sum() / len(df_jobs) * 100).round(2)
})
missing_df = missing_df[missing_df["Missing Count"] > 0].sort_values(by="Missing Count", ascending=False)
print("Columns With Missing Values:")
print(missing_df.to_string())

print("\nBasic Numerical Statistics:")
print(df_jobs[["salary", "views", "applies"]].describe().round(2).to_string())


Columns With Missing Values:
                   Missing Count  Missing Percentage (%)
skills_desc                15742                   99.09
med_salary                 14905                   93.82
remote_allowed             13546                   85.27
min_salary                 10365                   65.25
max_salary                 10365                   65.25
pay_period                  9384                   59.07
salary                      9384                   59.07
currency                    9384                   59.07
compensation_type           9384                   59.07
company_id                   366                    2.30
description                    1                    0.01

Basic Numerical Statistics:
           salary     views   applies
count     6502.00  15886.00  15886.00
mean     70187.78     63.42     12.50
std      77638.98    154.96     42.18
min         10.00      0.00      0.00
25%         31.29      2.00      0.00
50%      65000.00     15.00   

## 5. Data Preprocessing

The preprocessing module (`src/preprocessing.py`) transformed the raw, uncurated data:
1. **Deduplication**: Removed duplicate records based on `job_id`.
2. **Text Normalization**: Stripped excessive whitespace (`\s+` -> `' '`) and handled null descriptions.
3. **Imputation**: Missing titles and locations set to `'Unknown'`; missing experience levels set to `'Not Specified'`.
4. **Unified Salary Synthesis**: Synthesized `salary` from `med_salary` and average of `min_salary` and `max_salary`.
5. **Remote Status Standardization**: Transformed binary `remote_allowed` flags into categorical `'Remote'` and `'On-site'/'Not Specified'`.

Below is the verified audit summary generated during preprocessing:


In [11]:
prep_summary_path = "../outputs/results/preprocessing_summary.csv"
if os.path.exists(prep_summary_path):
    df_prep = pd.read_csv(prep_summary_path)
    print("Preprocessing Audit Summary (Before vs. After Cleaning):")
    print(df_prep.to_string(index=False))
else:
    print("Preprocessing summary file not found.")


Preprocessing Audit Summary (Before vs. After Cleaning):
                          Metric  Before_Preprocessing  After_Preprocessing
                      Total Rows                 15886                15886
                   Total Columns                    27                   23
               Duplicate Job IDs                     0                    0
     Missing (Null) Descriptions                     1                    0
       Missing (Null) Job Titles                     0                    0
        Missing (Null) Locations                     0                    0
Missing (Null) Experience Levels                  4902                    0
            Missing (Null) Views                  2763                    0
          Missing (Null) Applies                  7186                    0
     Jobs With Calculated Salary                  6502                 6502
          Remote Jobs Identified                  2340                 2340


## 6. Technical Skill Extraction

Module 3 (`src/skill_extraction.py`) defined an explainable taxonomy of 49 technical skills across 8 tech domains. Using boundary-safe regular expressions and tailored lookarounds for symbols like `C++`, `C#`, each job description was analyzed, counting each skill at most once per job posting.

Below we load and inspect the top extracted skills:


In [13]:
skills_path = "../outputs/results/top_skills.csv"
df_skills = pd.read_csv(skills_path)
print("Top 15 Most In-Demand Technical Skills:")
print(df_skills.head(15).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
top_15_skills = df_skills.head(15).iloc[::-1]
bars = ax.barh(top_15_skills["Skill"], top_15_skills["Job_Count"], color="#2b5c8f", height=0.65)
for bar, pct in zip(bars, top_15_skills["Demand_Percentage"]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2, f"{int(bar.get_width()):,} ({pct:.1f}%)", va='center', fontsize=9)
ax.set_title("Top 15 Technical Skills Ranked by Market Demand", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Job Postings Count")
ax.set_ylabel("Technical Skill")
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


Top 15 Most In-Demand Technical Skills:
                  Skill  Job_Count  Demand_Percentage
                  Excel       2596              16.34
                    SQL        764               4.81
                 Python        616               3.88
                    AWS        483               3.04
                   Java        403               2.54
                  Azure        390               2.45
             JavaScript        298               1.88
                  Linux        292               1.84
                   HTML        264               1.66
           Google Cloud        235               1.48
       Machine Learning        227               1.43
                  React        223               1.40
                Tableau        217               1.37
                    CSS        198               1.25
Artificial Intelligence        191               1.20


## 7. Job Market Analysis (EDA)

Exploratory analysis of labor market attributes generated by Module 4 (`src/analysis.py`):
- **Top Job Titles** (`top_job_titles.csv`)
- **Top Locations** (`top_locations.csv`)
- **Experience Levels** (`experience_levels.csv`)
- **Work Types** (`work_type.csv`)
- **Remote Work** (`remote_jobs.csv`)


In [15]:
print("--- Top 10 Job Titles ---")
df_titles = pd.read_csv("../outputs/results/top_job_titles.csv")
print(df_titles.head(10).to_string(index=False))

print("\n--- Top 10 Employment Locations ---")
df_locs = pd.read_csv("../outputs/results/top_locations.csv")
print(df_locs.head(10).to_string(index=False))

print("\n--- Experience Level Requirements ---")
df_exp = pd.read_csv("../outputs/results/experience_levels.csv")
print(df_exp.to_string(index=False))

print("\n--- Employment Work Types ---")
df_wt = pd.read_csv("../outputs/results/work_type.csv")
print(df_wt.to_string(index=False))

print("\n--- Remote Work Distribution ---")
df_remote = pd.read_csv("../outputs/results/remote_jobs.csv")
print(df_remote.to_string(index=False))


--- Top 10 Job Titles ---
                      Job_Title  Job_Count
Sales Director [Owner/Operator]         83
Sales Director {Owner/Operator}         78
                Sales Associate         65
         Retail Sales Associate         60
               Staff Accountant         42
                Project Manager         41
       Administrative Assistant         40
Customer Service Representative         39
            Executive Assistant         38
        Call Center Support Rep         38

--- Top 10 Employment Locations ---
       Location  Job_Count
  United States       1133
   New York, NY        398
    Chicago, IL        267
    Houston, TX        243
    Atlanta, GA        207
Los Angeles, CA        188
     Dallas, TX        182
     Austin, TX        168
     Boston, MA        147
  San Diego, CA        137

--- Experience Level Requirements ---
Experience_Level  Job_Count  Percentage
Mid-Senior level       5083       32.00
   Not Specified       4902       30.86
     Ent

## 8. Salary Analysis

Comprehensive compensation analysis based on the 6,502 postings with reported compensation. Missing salaries are never imputed with artificial numbers.

To prevent scale distortion caused by extreme multi-million dollar outliers during plotting, the histogram is clipped at the 99th percentile ($275,000) for visual clarity while the underlying data remains unaltered.


In [17]:
print("--- Overall Salary Descriptive Statistics ---")
df_sal_summary = pd.read_csv("../outputs/results/salary_summary.csv")
print(df_sal_summary.to_string(index=False))

print("\n--- Average & Median Salary by Experience Level ---")
df_sal_exp = pd.read_csv("../outputs/results/salary_by_experience.csv")
print(df_sal_exp.to_string(index=False))

print("\n--- Average & Median Salary by Work Type ---")
df_sal_wt = pd.read_csv("../outputs/results/salary_by_work_type.csv")
print(df_sal_wt.to_string(index=False))

valid_salaries = df_jobs["salary"].dropna()
valid_salaries = valid_salaries[(valid_salaries > 0) & np.isfinite(valid_salaries)]
p99 = valid_salaries.quantile(0.99)
vis_salaries = valid_salaries[valid_salaries <= p99]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(vis_salaries, bins=40, color="#2b5c8f", edgecolor="white", alpha=0.85)
median_sal = valid_salaries.median()
mean_sal = valid_salaries.mean()
ax.axvline(median_sal, color="red", linestyle="--", linewidth=2, label=f"Median: ${median_sal:,.0f}")
ax.axvline(mean_sal, color="orange", linestyle=":", linewidth=2, label=f"Mean: ${mean_sal:,.0f}")
ax.set_title("Salary Distribution (Clipped at 99th Percentile for Visualization)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Salary (USD)")
ax.set_ylabel("Number of Job Postings")
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


--- Overall Salary Descriptive Statistics ---
            Metric      Value
             Count    6502.00
              Mean   70187.78
            Median   65000.00
           Minimum      10.00
           Maximum 1050000.00
Standard Deviation   77638.98

--- Average & Median Salary by Experience Level ---
Experience_Level  Job_Count  Average_Salary  Median_Salary
Mid-Senior level       2149        87463.59       90000.00
   Not Specified       1937        66013.41       60000.00
     Entry level       1383        32322.68          30.38
       Associate        622        52147.08       52500.00
        Director        297       172759.95      180000.00
       Executive         65       177259.68      145000.00
      Internship         49        11517.56          35.05

--- Average & Median Salary by Work Type ---
 Work_Type  Job_Count  Average_Salary  Median_Salary
 Full-time       5404        82152.18       75900.00
  Contract        686         9787.64          47.50
 Part-time    

## 9. Data Visualizations Gallery

Below is the collection of exportable PNG figures generated by Module 5 (`src/visualization.py`) in `outputs/charts/`.


In [19]:
charts_folder = "../outputs/charts"
chart_names = [
    "top_skills.png",
    "top_job_titles.png",
    "top_locations.png",
    "experience_levels.png",
    "work_type.png",
    "remote_jobs.png",
    "salary_distribution.png",
    "salary_by_experience.png",
    "salary_by_work_type.png",
    "skill_demand.png"
]

for name in chart_names:
    path = os.path.join(charts_folder, name)
    if os.path.exists(path):
        print(f"Chart: {name:25s} Size: {os.path.getsize(path):,} bytes")
    else:
        print(f"Warning: Chart {name} not found.")


Chart: top_skills.png            Size: 88,106 bytes
Chart: top_job_titles.png        Size: 91,124 bytes
Chart: top_locations.png         Size: 82,240 bytes
Chart: experience_levels.png     Size: 65,102 bytes
Chart: work_type.png             Size: 62,326 bytes
Chart: remote_jobs.png           Size: 41,845 bytes
Chart: salary_distribution.png   Size: 52,091 bytes
Chart: salary_by_experience.png  Size: 64,843 bytes
Chart: salary_by_work_type.png   Size: 52,842 bytes
Chart: skill_demand.png          Size: 119,249 bytes


## 10. Machine Learning — Salary Prediction

Module 6 (`src/ml_model.py`) trains an explainable **RandomForestRegressor** to predict annual compensation based on non-leaking role metadata:
- `formatted_experience_level`
- `formatted_work_type`
- `remote_status`
- `clean_location` (top metro hubs + 'Other')
- `job_category` (derived from job title)

The dataset used for ML comprises 3,906 validated annual compensation records ($10,000 to $500,000), excluding hourly rates (< $1,000) to maintain scale consistency.

Below we load the model results and demonstrate inference with the serialized Scikit-learn Pipeline (`outputs/results/salary_model.pkl`):


In [21]:
print("--- Machine Learning Model Results ---")
df_ml_results = pd.read_csv("../outputs/results/ml_results.csv")
print(df_ml_results.to_string(index=False))

print("\n--- Testing Model Pipeline Inference ---")
model_path = "../outputs/results/salary_model.pkl"
model = joblib.load(model_path)

sample_postings = pd.DataFrame([
    {
        "formatted_experience_level": "Mid-Senior level",
        "formatted_work_type": "Full-time",
        "remote_status": "Remote",
        "clean_location": "New York, NY",
        "job_category": "Software & Engineering"
    },
    {
        "formatted_experience_level": "Entry level",
        "formatted_work_type": "Full-time",
        "remote_status": "Not Specified",
        "clean_location": "Chicago, IL",
        "job_category": "Data & Analytics"
    },
    {
        "formatted_experience_level": "Director",
        "formatted_work_type": "Full-time",
        "remote_status": "Remote",
        "clean_location": "San Francisco, CA",
        "job_category": "Management & Leadership"
    }
])

predictions = model.predict(sample_postings)
sample_postings["Predicted_Salary_USD"] = [f"${p:,.2f}" for p in predictions]
print(sample_postings.to_string(index=False))


--- Machine Learning Model Results ---
          Metric                                                                                        Value
           Model                                                                        RandomForestRegressor
Training Samples                                                                                         3124
 Testing Samples                                                                                          782
   Features Used formatted_experience_level, formatted_work_type, remote_status, clean_location, job_category
             MAE                                                                                     35525.24
            RMSE                                                                                     49624.87
              R2                                                                                       0.2699

--- Testing Model Pipeline Inference ---
formatted_experience_level formatted_wo

## 11. Personalized Career Intelligence & Skill Roadmap System

SkillSight extends macro-level job market telemetry into an individual career development engine.
By integrating candidate academic records, self-assessed technical proficiencies, and career goals with the verified dataset, SkillSight computes:
1. **Deterministic Skill-Gap Analysis** (Already Acquired vs. Skills to Develop with exact dataset demand percentages)
2. **Analytical Career Readiness Scoring** (0–100 decomposition across skill coverage, depth, academics, and projects)
3. **Personalized 5-Phase Career Roadmaps** (Structured learning sequences tailored around identified gaps)
4. **Tailored Portfolio Projects** (Rule-based project specifications mapped to target roles)
5. **Multi-Role Skill Alignment** (Compatibility percentages across 14 distinct career tracks)


In [23]:
# Import the SkillSight personalization modules
import sys
sys.path.append("..")
from src.database import init_db
from src.user_profile import get_user_profile
from src.career_matching import match_career_roles
from src.skill_gap import analyze_skill_gap
from src.personalization import calculate_career_readiness, get_what_to_learn_next, get_career_market_alerts
from src.roadmap import generate_career_roadmap
from src.project_recommender import recommend_projects
from src.progress_tracker import get_user_progress_summary

# 1. Initialize SQLite database & inspect demo candidate profile
init_db("../data/skillsight.db")
demo_profile = get_user_profile(1)

print("=== CANDIDATE PROFILE ===")
print(f"Name:        {demo_profile['personal']['name']}")
print(f"Academic:    {demo_profile['personal']['degree']} in {demo_profile['personal']['branch']} ({demo_profile['personal']['academic_year']})")
print(f"CGPA:        {demo_profile['personal']['cgpa']}")
print(f"Career Goal: {demo_profile['preferences']['preferred_role']} ({demo_profile['preferences']['work_mode']})")
print(f"Skills:      {', '.join([s['skill_name'] + ' (' + s['proficiency'] + ')' for s in demo_profile['skills']])}")

# 2. Skill-Gap Analysis
target_role = demo_profile['preferences']['preferred_role']
user_skills = demo_profile['skills']
gap_results = analyze_skill_gap(user_skills, target_role)

print(f"\n=== SKILL GAP ANALYSIS: {target_role.upper()} ===")
print(f"Role Skill Coverage: {gap_results['coverage_percentage']}%")
print(f"Already Acquired ({len(gap_results['already_have'])}): {', '.join([s['skill'] for s in gap_results['already_have']])}")
print("\nTop Missing Skills to Develop:")
for item in gap_results['skills_to_develop'][:5]:
    print(f"  → #{item['learning_order']}: {item['skill']} | Priority: {item['priority']} | Dataset Demand: {item['demand_percentage']:.2f}% ({item['job_count']:,} jobs)")

# 3. Career Readiness Score
readiness = calculate_career_readiness(demo_profile, user_skills, target_role)
print(f"\n=== CAREER READINESS SCORE: {readiness['readiness_score']} / 100 ({readiness['status_label']}) ===")
for factor, val in readiness['factors'].items():
    print(f"  • {factor.replace('_', ' ').title()}: {val['score']} / {val['max']} pts ({val['percentage']}%)")

# 4. 'What Should I Learn Next?'
next_skills = get_what_to_learn_next(user_skills, target_role)
print("\n=== 'WHAT SHOULD I LEARN NEXT?' (TOP 3 RECOMMENDATIONS) ===")
for i, n in enumerate(next_skills, 1):
    print(f"{i}. {n['skill']} ({n['priority']} Priority) - {n['reason']}")

# 5. Tailored Portfolio Projects
projects = recommend_projects(user_skills, target_role)
print("\n=== RECOMMENDED PORTFOLIO PROJECTS ===")
for p in projects[:2]:
    print(f"📁 {p['title']} [{p['difficulty']}]")
    print(f"   Target Skills: {', '.join(p['skills_developed'])}")
    print(f"   Outcome: {p['outcome']}")

# 6. Multi-Role Skill Alignment
role_matches = match_career_roles(user_skills)
print("\n=== MULTI-ROLE SKILL ALIGNMENT (TOP 5) ===")
for m in role_matches[:5]:
    print(f"• {m['role']:<25} Alignment: {m['alignment_percentage']}% ({m['matched_count']}/{m['total_count']} skills matched)")


=== CANDIDATE PROFILE ===
Name:        Alex Morgan
Academic:    B.Tech in Computer Science & Engineering (3rd Year)
CGPA:        8.65
Career Goal: Data Scientist (Hybrid)
Skills:      Excel (Advanced), Git (Beginner), Java (Intermediate), NumPy (Beginner), Pandas (Intermediate), Python (Intermediate), SQL (Intermediate)

=== SKILL GAP ANALYSIS: DATA SCIENTIST ===
Role Skill Coverage: 33.3%
Already Acquired (4): Python, SQL, Pandas, NumPy

Top Missing Skills to Develop:
  → #1: AWS | Priority: MEDIUM | Dataset Demand: 3.04% (483 jobs)
  → #2: Machine Learning | Priority: MEDIUM | Dataset Demand: 1.43% (227 jobs)
  → #3: Scikit-learn | Priority: MEDIUM | Dataset Demand: 0.16% (25 jobs)
  → #4: Tableau | Priority: LOW | Dataset Demand: 1.37% (217 jobs)
  → #5: Docker | Priority: LOW | Dataset Demand: 0.86% (137 jobs)

=== CAREER READINESS SCORE: 61 / 100 (Competitive Progress) ===
  • Skill Coverage: 15.0 / 45.0 pts (33.3%)
  • Proficiency Depth: 13.8 / 20.0 pts (68.8%)
  • Academic Align

## 12. Results & Key Findings

All figures are grounded directly in the verified results generated across Modules 1–6:

1. **Macro Dataset Scale**: 15,886 unique job postings across 10,903 distinct roles, 3,010 geographic locations, and 6,030 hiring companies.
2. **Top Technical Skills**:
   - **Excel**: #1 overall tool with **2,596 postings (16.34%)**.
   - **SQL**: #1 database competency with **764 postings (4.81%)**.
   - **Python**: #1 programming language with **616 postings (3.88%)**.
   - **Cloud Computing**: **AWS (483, 3.04%)** and **Azure (390, 2.45%)** demonstrate strong multi-cloud enterprise demand.
   - **Web & Software**: **JavaScript (298, 1.88%)**, **HTML (264, 1.66%)**, **React (223, 1.40%)**.
3. **Experience Level Demand**: Mid-Senior level positions dominate the market (**32.00%**), followed by Entry level (**23.25%**).
4. **Work Types**: Full-time arrangements represent the vast majority (**80.85%**), while Contract work constitutes **10.95%**.
5. **Remote Opportunities**: **14.73% (2,340 postings)** are remote positions.
6. **Compensation**: Across the 6,502 postings reporting salary, the overall mean is **$70,188** and median is **$65,000**. Full-time salaries average **$82,152**, whereas Director positions average **$172,760**.
7. **Predictive Modeling**: The Random Forest regressor achieves **MAE = $35,525**, **RMSE = $49,625**, and **R² = 0.2699** on unseen test data.


## 13. Limitations

Documented constraints and realistic considerations of the current system:
1. **Missing Salary Information**: Only 6,502 out of 15,886 records (40.9%) include compensation data, introducing potential reporting self-selection bias.
2. **Mixed Compensation Schedules**: LinkedIn postings report compensation under hourly, monthly, and yearly schedules, necessitating strict filtering for annual modeling.
3. **Salary Outliers**: Extreme executive figures (up to $1,050,000) and data entry errors create wide variance.
4. **Predefined Skill Dictionary**: Skill detection relies on a curated 49-skill dictionary; emerging or niche tools outside the dictionary are not detected.
5. **Dataset Representation**: Scraped listings represent a snapshot of 2023 postings, weighted toward corporate and technical sectors.
6. **Machine Learning Predictive Power**: With $R^2 \approx 0.27$, high-level metadata accounts for roughly a quarter of salary variance, as compensation is heavily influenced by unmeasured factors (firm size, equity, specific tech depth, negotiation).


## 14. Conclusion

SkillSight successfully delivers an integrated, explainable end-to-end Data Science project:
- **Clean Pipelines**: Robust data cleaning and deterministic regex-based skill extraction without reliance on black-box LLMs or external APIs.
- **Actionable Telemetry**: Comprehensive EDA and visualization capturing skill demand, career seniority trends, and compensation patterns.
- **Operational ML Pipeline**: Serialized Random Forest model capable of real-time salary estimation based on role attributes.
- **Interactive Delivery**: Fully integrated Streamlit web application providing academic transparency and accessibility to non-technical users.
